# Transformer と LLM の学習・文章生成



## 1. Transformerとは

Transformer は、

```text id="jlywmde"
token 同士の関係を
attention で学習するモデル
```

です。

現在の多くの LLM の基礎になっています。






## 2. 入力

文章を token 化：

\[
X=[x_1,x_2,\dots,x_L]
\]

* (L)：token 数
* (x_i)：token

です。





## 3. embedding

各 token をベクトル化：

\[
e_i=E(x_i)
\]

* $E$：embedding table
* $e_i\in\mathbb{R}^d$

です。





## 4. 位置情報

順番を表すため：

\[
h_i=e_i+p_i
\]

を作ります。

* $p_i$：position embedding





## 5. 行列化

全 token を並べる：

\[
H
\in
\mathbb{R}^{L\times d}
\]

これが Transformer 入力。





## 6. Query, Key, Value

attention 用に変換：

\[
Q=HW_Q
\]

\[
K=HW_K
\]

\[
V=HW_V
\]

です。





## 7. self-attention

類似度：

\[
QK^\top
\]

を計算。

\[
(q_i^\top k_j)
\]

は：

```text id="jlywmdf"
token i が
token j をどれくらい見るか
```

です。







## 8. attention

まず token 間の similarity：

\[
S
=

QK^\top
\]

を計算する。

ここで、

\[
S_{ij}
=

q_i^\top k_j
\]

は、

```text id="jlwmup9"
token i と token j の類似度
```

を表す。

次に：

\[
A
=

\mathrm{softmax}
\left(
\frac{S}{\sqrt d}
\right)
\]

を計算する。
これは similarity を確率的重みへ変換する操作であり、
この式自体は新規ではない。


最後に、

新しい token 表現：

\[
H'
=

AV
\]

を計算する。

ここで：

* \(V\)：Value 行列
* \(A_{ij}\)：token ($i$) が token ($j$) をどれくらい参照するか

を表す。

したがって、

```text id="jlwmuqa"
重要な token の情報を
重み付き平均で集約
```

している。




## 9. GPT型



GPT は次token予測：

\[
P(x_t\mid x_{<t})
\]
をします。

このときに未来 token を見ないよう：
causal mask
```text id="jlywme8"
token i から j>i である
未来 token j>i を
見えなくする
```
を行う。


#### causal mask 前

時間$i$→$j$を見るとして、

```text id="jlywmed"
1→1 1→2 1→3
2→1 2→2 2→3
3→1 3→2 3→3
```


#### causal mask 後

```text id="jlywmee"
1→1
2→1 2→2
3→1 3→2 3→3
```

のみ参照する。


式で書き直すと
mask 前：

$$
S=
\left[
\begin{array}{ccc}
s_{11} & s_{12} & s_{13} \\
s_{21} & s_{22} & s_{23} \\
s_{31} & s_{32} & s_{33}
\end{array}
\right]
$$

mask 後：

$$
\tilde S=
\left[
\begin{array}{ccc}
s_{11} & -\infty & -\infty \\
s_{21} & s_{22} & -\infty \\
s_{31} & s_{32} & s_{33}
\end{array}
\right]
$$

である。expを作用させる$A$は、$-\infty $部分が0となる。










## LLM の学習と文章生成

Ref. 
[Vaswani2017attention] Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, Ł., & Polosukhin, I. Attention Is All You Need. Advances in Neural Information Processing Systems 30 (NeurIPS 2017). arXiv.1706.03762.



## 記号一覧

| 記号           | 意味                        |
| ------------ | ------------------------- |
| ($x_t$)        | t番目のtoken                 |
| ($e_t$)        | token embedding           |
| ($h_t$)        | hidden state（文脈込み内部状態）    |
| ($z_i$)        | token (i) の出力スコア（logit）   |
| ($P(x_{t+1})$) | 次token確率                  |
| ($E$)          | embedding行列               |
| ($V$)          | 語彙数（vocabulary size）      |
| ($d$)          | embedding次元               |
| ($p_t$)        | 位置情報（position embedding）  |
| ($Q,K,V$)      | Attention用Query/Key/Value |
| ($d_i$)        | i番目の文書（document）          |
| ($v_i$)        | 文書embedding               |
| ($T$)          | temperature（温度）           |



## 1. embeddingとは

embeddingとは、

> token（単語・部分単語）を数値ベクトルへ変換したもの

です。

LLMは文字列を直接処理せず、

$$
x_t
\rightarrow
e_t
$$

として内部計算する。

* ($x_t$)：token
* ($e_t$)：embeddingベクトル

です。



## embedding行列（単語辞書）

LLMは：

$$
E \in \mathbb{R}^{V \times d}
$$

という embedding 行列を持つ。

* (V)：語彙数(token辞書サイズ)
* (d)：embedding次元

です。

各tokenは：

$$
e_t = E[x_t]
$$

で取得される。

つまり：

> token ↔ embedding の巨大辞書

を持っている。



## 順方向変換

入力token：

$$
x_t
$$

から embedding：

$$
e_t
$$

を取得する。

$$
x_t
\rightarrow
e_t
$$

これが順方向変換。



## 逆方向変換（文章生成）

LLM内部では：

$$
h_t
$$

という hidden state を作る。

これは：

> token ($x_t$) の文脈込み内部状態

です。

生成時は：

$$
z_i = e_i^T h_t
$$

を計算。

* ($e_i$)：token ($i$) のembedding
* ($h_t$)：現在状態

です。

これは：

> token ($i$) が次に来るらしさ

を表す。

次に：

$$
P(x=i)=\frac{e^{z_i}}{\sum_j e^{z_j}}
$$

を計算し（全ての$i$です)、
次token確率を得る。

つまり：

$$
h_t
\rightarrow
P(x_{t+1})
$$

という逆方向変換をしている。



## weight tying

多くのLLMでは：

* 入力embedding
* 出力側unembedding

で同じ embedding 行列：

$$
E
$$

を共有する。

これを：

> weight tying

という。



## 2. embeddingとTransformerを用いた学習と文章生成



## token列

文章：

```text id="jlwm8m"
I love cats
```

を：

$$
x_1,x_2,x_3
$$

とする。



## embedding

各tokenを embedding 化：

$$
e_t = E(x_t)
$$

する。

さらに位置情報：

$$
p_t
$$

を加える。

$$
h_t^{(0)}=e_t + p_t
$$



## Transformer

Transformerでは：

$$
h_t^{(l)}
$$

を：

> 第 (l) 層での token (t) の hidden state

とする。

Transformerは：

$$
h_t^{(l+1)}=f(h_1^{(l)},\dots,h_n^{(l)})
$$

として、

全token状態を同時更新する。



## Self-Attention

Self-attentionでは：

$$
\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\left(\frac{QK^T}{\sqrt{d}}\right)V
$$

を計算。

各tokenが：

> 他の全tokenを参照

する。



## 学習過程

LLMは：

$$
P(x_{t+1}\mid x_{\le t})
$$

を最大化するよう学習する。



## hidden state

Transformerにより：

$$
h_t = f(x_1,x_2,\dots,x_t)
$$

を作る。

$$
h_t
$$

には：

* 現在token
* 過去token
* 文脈

が含まれる。



## 次token確率

hidden state：

$$
h_t
$$

から：

$$
z_i = e_i^T h_t
$$

を計算。

次に：

$$
P(x_{t+1}=i \mid x_{\le t})=\frac{
e^{z_i}
}{
\sum_j e^{z_j}
}
$$

を計算する。



## loss

正解token：

$$
x_{t+1}^{\mathrm{true}}
$$

に対して：

$$
L=-\log P(
x_{t+1}^{\mathrm{true}}
\mid
x_{\le t}
)
$$

を最小化する。

これを全token位置で繰り返す。



## 学習の流れ

### 1. token入力

$$
x_1,x_2,\dots,x_t
$$



## 2. embedding

$$
x_t \rightarrow e_t
$$



## 3. Transformer

$$
e_1,\dots,e_t
\rightarrow
h_t
$$



## 4. 次token確率

$$
h_t
\rightarrow
P(x_{t+1})
$$



## 5. loss計算

正解tokenとの誤差を計算。



## 6. 重み更新

embedding行列：

$$
E
$$

や Transformer重みを gradient descent で更新。



## 文章生成

生成時も：

$$
h_t
\rightarrow
P(x_{t+1})
$$

を計算する。



## 温度（temperature）

temperature：

$$
T
$$

を使うと：

$$
P(x=i) =
\frac{
e^{z_i/T}
}{
\sum_j e^{z_j/T}
}
$$

となる。

### 低温 (T<1)

* 高確率tokenを強調
* deterministic(毎回同じ出力、例：T=0の場合)

### 高温 (T>1)

* ランダム性増加
* 多様性増加



## token選択

確率分布から：

$$
x_{t+1}
$$

を選択。

その後：

$$
x_{t+1}
\rightarrow
\mathrm{embedding}
\rightarrow
\mathrm{Transformer}
\rightarrow
P(x_{t+2})
$$

を繰り返す。



## 本質

LLMは：

$$
x_t
\rightarrow
e_t
\rightarrow
h_t
\rightarrow
P(x_{t+1})
$$

という流れで、

embedding空間上で文脈状態：

$$
h_t
$$

を更新しながら、

次token確率を生成する巨大確率モデルです。


---



検索用の埋め込みは、文章同士の類似度を比べるためにも使われます。検索への応用は [RAG](8300.0150.RAG.ipynb) を参照してください。

## 10. context window

Transformer は、メモリー量・計算量の制限により：
```text id="jlywmdj"
一度に見られるtoken数
```

に上限があります。

これが：

```text id="jlywmdk"
context window
```

です。


### 入力文と token の対応

例えば入力文を：

```text
私は 学生 です
```

とします。

token 化された結果を簡単化して：

\[
x_1=\text{「私は」},\quad
x_2=\text{「学生」},\quad
x_3=\text{「です」}
\]

と書きます。

したがって入力列は：

\[
X=[x_1,x_2,x_3]
=

[\text{「私は」},\text{「学生」},\text{「です」}]
\]

です。



## context window との対応

context window が最大 ($L_{\max}$) token だとすると、入力できる token 列は：

\[
X=[x_1,x_2,\dots,x_L],
\qquad
L\le L_{\max}
\]

です。

$L_{\max}=3$とすると、
\(
X=[x_1,x_2,x_3],
\)
\(
X=[x_2,x_3,x_4],
\)
\(
X=[x_3,x_4,x_5]
\)
のように「$x_i$の範囲=Context Windowをずらしながら」transformerに入力します。

（system promptは必ず含まれます。）


### RNN
RNNは
$P(x_3|x_2)P(x_2|x_1)P(x_1)$
と逐次生成します。

### Transformer

一方Transformerは

\[
X=[x_1,x_2,x_3]
\]

を embedding：

\[
H=[h_1,h_2,h_3]
\]
に変換して、出力

\[
h_3'
=

A_{31}v_1
+
A_{32}v_2
+
A_{33}v_3
\]
として、

\[
P(x_4\mid x_1,x_2,x_3)
=

\mathrm{softmax}(W_oh_3')
\]

を$x_1,x_2,x_3$のhidden stateを並列に計算します。
**このため高速になります。**

Transformer では、$QK^⊤$の大規模行列計算を並列実行します。GPU はこのような並列計算に適しており、HBM（High Bandwidth Memory）は、attention 計算で必要となる大量の行列データ転送を高速化するために用いられるます。




## 最大context window

browser に入力可能な文字数が制限されるが、これは
**最大 context window そのものではない。**

context window とは、

```text id="jlwmuql"
System
User(#1)
Assistantr(#1)
Userr(#2)
Assistantr(#2)
Userr(#3)
...
```

のような会話履歴全体を token 化した総量である。

つまり、

* system prompt
* user 入力
* assistant 出力
* tool 出力
* memory

などを含めた、

\[
X=[x_1,x_2,\dots,x_L]
\]

全体の token 数 (L) が context window に対応する。
(正確には$x_t$を$h_t$に変換する。)






## 参考文献

[Vaswani2017attention] Vaswani, A., Shazeer, N., Parmar, N., et al. Attention Is All You Need. Advances in Neural Information Processing Systems 30 (NeurIPS 2017). DOI: 10.48550/arXiv.1706.03762.




# 長い文脈で情報を使う難しさ

LLM が長いプロンプトや多ターン対話で、以前の情報を忘れたり整合性を失ったりする問題については、近年多く研究されています。




## 1. Multi-turn conversation での忘却

“LLMs Get Lost in Multi-Turn Conversation” は、長い会話で LLM が以前の指示や状態を失う問題を分析した研究です。

主な観察：

* 会話ターンが増えると、

  * 過去の制約を無視する
  * 初期 instruction を忘れる
  * 以前の発言と矛盾する
  * 会話状態を誤認する

という現象が増加する。

重要なのは：

* context window 内に情報が存在していても忘れる
* token 長だけでなく attention 分散が本質
* 長い会話では state tracking が崩れる

という点です。

これは単なる「記憶容量不足」ではなく、

```text id="u6m4dz"
多数の token 間で attention が競合する
```

ことが問題だと解釈できます。

参考文献：

[LLMsGetLostMultiTurn2025] Li, X., et al. LLMs Get Lost in Multi-Turn Conversation. arXiv preprint, 2025. DOI: 10.48550/arXiv.2505.06120.





## 2. Long context で中央情報が失われる問題

Lost in the Middle: How Language Models Use Long Contexts

この研究では、長文 context 内のどこにある情報が利用されやすいかを調査しています。

結果：

* 先頭付近 → 利用されやすい
* 末尾付近 → 利用されやすい
* 中央付近 → 利用されにくい

つまり：

```text id="0kdy54"
[重要情報]
...
[大量の途中情報]
...
[現在の質問]
```

のようになると、中間部分の情報が脱落しやすい。

これは “Lost in the Middle” 問題と呼ばれます。

参考文献：

[LostInTheMiddle2024] Liu, Nelson F., et al. Lost in the Middle: How Language Models Use Long Contexts. Transactions of the Association for Computational Linguistics, 2024. DOI: 10.1162/tacl_a_00638.




## 3. Recency Bias（最近情報優先）

“Recency Bias in Large Language Models” は、LLM が後半 token を優先しやすいことを分析した研究です。

主な内容：

* 後ろ側の token が強く参照される
* 古い instruction が弱くなる
* 直近の情報に引きずられる

結果として：

```text id="o3qjtm"
最初の system prompt
↓
長い会話
↓
最近の user 発話が優先
```

となりやすい。

これは実務上非常に重要で、

* policy drift
* instruction drift
* role drift

の原因になります。

参考文献：

[RecencyBiasLLM2024] Zhang, Y., et al. Recency Bias in Large Language Models. arXiv preprint, 2024. DOI: 10.48550/arXiv.2402.14820.





## 4. なぜ最初と最後が強いのか

Transformer 系 LLM では、attention により token 間の関連度を計算します。

$$
\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$

長い context では：

* attention competition
* context dilution
* recency bias

が起きます。

さらに、学習データ側にも偏りがあります。





## 5. Fine tuning / instruction tuning による位置バイアス

実際の instruction tuning や RLHF 用データでは、

```text id="0n4u17"
最初：
重要 instruction
system prompt
task definition

最後：
最終質問
回答対象
```

が置かれることが非常に多い。

そのためモデルは学習中に：

```text id="6bzy6j"
「先頭と末尾が重要」
```

という統計的傾向を強く学習します。

結果として：

* 中央 token を軽視
* 最後の instruction を優先
* 古い制約を忘却

しやすくなります。

つまり “Lost in the Middle” や “Recency Bias” は、

* Transformer attention の構造
  だけでなく、
* fine tuning データ分布

によっても強化されていると考えられます。





## 6. 要約学習（summarization training）による先頭・末尾バイアス

文章要約データでも、
「良い」文章というのは
```text id="7r7q7x"
冒頭：
主題・結論・問題設定

末尾：
まとめ・結論
```

が配置されることが多い。

ニュース記事、
論文、
解説文、
ブログ記事などでは：

* 冒頭に概要
* 最後に結論

を書く構造が一般的です。

そのため要約タスクを大量学習すると、モデルは：

```text id="72m3g8"
重要情報は最初と最後にある
```

という統計的傾向も学習します。

結果として：

* 中央部分の情報を軽視
* 前半と後半を優先
* 中間の条件や例外を落とす

傾向が強化されます。

これは “Lost in the Middle” をさらに悪化させる一因と考えられます。





## 7. 数理的な見方

LLM は本質的に：

$$
P(x_t \mid x_{<t})
$$


を逐次最大化しているだけです。

つまり：

* 永続 state を持たない
* 会話状態を厳密保存しない
* 毎 token ごとに再計算

です。

そのため：

```text id="o0um0r"
古い情報
vs
最近の大量 token
```

の attention 競合で、
古い制約が埋もれます。




## 8. なぜ Agent が必要になるか

この問題を避けるために：

* external memory
* retrieval
* state management
* harness engineering

が重要になります。

つまり：

```text id="pwwm3u"
全部を prompt に押し込む
↓
破綻
```
```
必要情報だけ再取得
↓
状態を外部化
```

という方向に移っています。


---



# 無限メモリー


Claudeでは、長い会話中に

```text id="jlwmuqo"
会話を要約・圧縮しています
```

というメッセージが表示されます。

これは、

```text id="jlwmuqp"
会話履歴全体を
context window 内へ収める
```

ために行われている。

Claude のこの動作は、

```text id="jlwmuqq"
アプリケーション側の summary memory
```

であり、Transformer 内部の attention そのものではない。

一方で、

```text id="jlwmuqr"
Transformer 内部で
過去情報を圧縮保持する
```

研究も存在する。

代表例として、

Leave No Context Behind: Efficient Infinite Context Transformers with Infini-attention

では、

```text id="jlwmuqs"
古い token列を
固定サイズ memory へ圧縮
```

することで、事実上の infinite context を実現しようとしている。




## 概要

Leave No Context Behind: Efficient Infinite Context Transformers with Infini-attention は、

> Transformer に「事実上の無限長コンテキスト」を持たせる

ことを目的とした論文です。

従来 Transformer の問題は：

* attention 計算量が
  \[
  O(L^2)
  \]
  （(L)：系列長, TOKEN長）
* KV cache が線形増加
* context window を超えると忘却

という点でした。

この論文は：

```text
局所 attention
+
圧縮メモリ(memory)
```

を組み合わせることで、

```text
有限計算量で
非常に長い文脈を扱う
```

ことを狙っています。





## 従来 Transformer の問題

通常の self-attention は：

\[
\text{memory}
\sim O(L^2)
\]

になります。

そのため：

* 1M token
* 長時間会話
* 長いコードベース
* 書籍全体

などでは破綻します。





## 基本アイデア

Infini-attention の核心は：

```text
遠距離文脈を
圧縮メモリへ蓄積する
```

ことです。

つまり：

```text
近距離
→ 通常 attention

遠距離
→ 圧縮 memory
```

へ分離します。





## chunk 化

入力列を chunk に分割します。

例えば：

```text
x1 ... x1024
x1025 ... x2048
x2049 ...
```

のように処理。

各 chunk 内では普通の attention を行います。





## memory の導入

過去 chunk 全体をそのまま保持すると：

```text
memory explosion
```

になります。

そこで：

```text
過去情報を固定サイズ memory に圧縮
```

します。






## 「Infinite Context」とは？

もちろん物理的無限ではありません。

意味としては：

```text
context lengthに対して
計算量・メモリ量が発散しにくい
```

ということです。

従来：

\[
\text{cost}\sim O(L^2)
\]

Infini-attention：

\[
\text{cost}\sim O(L)
\]

に近づけます。




## 重要な点

この論文の本質は：

```text
「全tokenを保持する」
というTransformer原理を捨て始めた
```

ことです。

従来 Transformer は：

```text
全履歴を exact に保持
```

していました。

Infini-attention は：

```text
古い履歴を圧縮近似する
```

方向へ進んでいます。


人間の記憶も、短期的に保持する記憶と、長期間保持する記憶に大別されます。LLMはそれと近い進化をしています。


初期の画像認識ニューラルネットワーク、特に Convolutional Neural Network (CNN) は、人間の視覚野を参考に設計されました。特に、David Hubel と Torsten Wiesel は、「視覚野ニューロンが局所的な線やエッジへ反応する」ことを発見しました。CNN の convolution filter は、この局所特徴抽出の考え方に対応しています。ただし現在の深層学習は、脳を正確に再現しているわけではなく、工学的に発展したモデルです。



## 何が嬉しいか

### 1. 長い会話

数十万 token の会話。


### 2. 長いコードベース

巨大 repository 理解。



### 3. 書籍全体

全文読解。



### 4. agent memory

長期状態保持。





## 限界

当然ながら：

```text
圧縮すると情報損失
```

があります。

つまり：

* 完全記憶ではない
* retrieval error がある
* 古い情報がぼやける

という問題があります。





## 参考文献

[LeaveNoContextBehind2024infini] Munkhdalai, T., Yu, M., Kora, S., et al. Leave No Context Behind: Efficient Infinite Context Transformers with Infini-attention. arXiv preprint arXiv:2404.07143 (2024). DOI: 10.48550/arXiv.2404.07143.

[Hubel1962receptivefield] Hubel, D. H., & Wiesel, T. N. Receptive fields, binocular interaction and functional architecture in the cat's visual cortex. Journal of Physiology, 160(1), 106–154 (1962). DOI: 10.1113/jphysiol.1962.sp006837.
